# Object Detection Project

## 1. Settings

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/HoangDuonng1359/object-detection"
PROJECT_DIR = Path("/kaggle/working/object-detection")

IMAGE_SIZE = 640
EPOCHS = 100
BATCH_SIZE = 8
NUM_WORKERS = 2
LR = 1e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
USE_PRETRAINED_BACKBONE = True
FREEZE_BACKBONE_STEM = False
OVERSAMPLE_CLASSES = ["chair"]
OVERSAMPLE_FACTOR = 2.0

# Set these to small values for a quick smoke test. Use 0 for full train/validation.
MAX_TRAIN_BATCHES = 0
MAX_VAL_BATCHES = 0

CONF_THRESHOLD = 0.25
NMS_THRESHOLD = 0.35

# Optional hidden/test image directory. Leave as None if you only want val predictions.
TEST_IMAGE_DIR = None
FINAL_PREDICTIONS = Path("/kaggle/working/predictions.json")


## 2. Clone Code And Install Requirements

In [2]:
import os
import subprocess
import sys

if not PROJECT_DIR.exists():
    if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
        raise ValueError("Edit REPO_URL in the settings cell before cloning.")
    clone_cmd = ["git", "clone"]
    clone_cmd += [REPO_URL, str(PROJECT_DIR)]
    subprocess.run(clone_cmd, check=True)
else:
    print(f"Project directory already exists: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())

requirements = PROJECT_DIR / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)

import torch
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())


Cloning into '/kaggle/working/object-detection'...


cwd: /kaggle/working/object-detection
python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch: 2.10.0+cu128
cuda_available: True


## 3. Locate Kaggle Input Dataset

In [3]:
PUBLIC_DIR = Path("/kaggle/input/datasets/duong1589/object-detection/public")
TRAIN_JSON = PUBLIC_DIR / "annotations" / "train.json"
VAL_JSON = PUBLIC_DIR / "annotations" / "val.json"
TRAIN_IMAGES = PUBLIC_DIR / "train" / "images"
VAL_IMAGES = PUBLIC_DIR / "val" / "images"
EVALUATOR = PUBLIC_DIR / "tools" / "evaluate_predictions.py"

print("PUBLIC_DIR:", PUBLIC_DIR)
print("TRAIN_JSON:", TRAIN_JSON)
print("VAL_JSON:", VAL_JSON)
print("TRAIN_IMAGES:", TRAIN_IMAGES)
print("VAL_IMAGES:", VAL_IMAGES)
print("EVALUATOR:", EVALUATOR, EVALUATOR.exists())


PUBLIC_DIR: /kaggle/input/datasets/duong1589/object-detection/public
TRAIN_JSON: /kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json
VAL_JSON: /kaggle/input/datasets/duong1589/object-detection/public/annotations/val.json
TRAIN_IMAGES: /kaggle/input/datasets/duong1589/object-detection/public/train/images
VAL_IMAGES: /kaggle/input/datasets/duong1589/object-detection/public/val/images
EVALUATOR: /kaggle/input/datasets/duong1589/object-detection/public/tools/evaluate_predictions.py True


## 4. Sanity Check Dataset And Code

In [4]:
import json

for split_name, json_path in [("train", TRAIN_JSON), ("val", VAL_JSON)]:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    print(split_name, "images=", len(data["images"]), "annotations=", len(data["annotations"]))

subprocess.run([sys.executable, "-m", "py_compile", "train.py", "predict.py"], check=True)
subprocess.run([
    sys.executable, "-m", "utils.check_dataset",
    "--annotations", str(TRAIN_JSON),
    "--image_dir", str(TRAIN_IMAGES),
    "--image_size", str(IMAGE_SIZE),
    "--batch_size", "2",
    "--train",
], check=True)


train images= 7500 annotations= 10642
val images= 1500 annotations= 2021
dataset_size=7500
classes=['person', 'car', 'dog', 'cat', 'chair']
sample_image_shape=(3, 640, 640)
sample_image_id=img_000c52c6d12f.jpg
sample_boxes=(1, 4)
batch_image_shape=(2, 3, 640, 640)
batch_box_counts=[1, 1]


CompletedProcess(args=['/usr/bin/python3', '-m', 'utils.check_dataset', '--annotations', '/kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json', '--image_dir', '/kaggle/input/datasets/duong1589/object-detection/public/train/images', '--image_size', '640', '--batch_size', '2', '--train'], returncode=0)

## 5. Train

In [5]:
from IPython.display import display
import pandas as pd

train_cmd = [
    sys.executable, "train.py",
    "--train_data", str(TRAIN_JSON),
    "--val_data", str(VAL_JSON),
    "--image_dir", str(TRAIN_IMAGES),
    "--val_image_dir", str(VAL_IMAGES),
    "--checkpoint_dir", str(PROJECT_DIR / "models"),
    "--image_size", str(IMAGE_SIZE),
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--lr", str(LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--conf_threshold", str(CONF_THRESHOLD),
    "--nms_threshold", str(NMS_THRESHOLD),
]
if USE_AMP:
    train_cmd.append("--amp")
if USE_PRETRAINED_BACKBONE:
    train_cmd.append("--pretrained_backbone")
if FREEZE_BACKBONE_STEM:
    train_cmd.append("--freeze_backbone_stem")
if OVERSAMPLE_CLASSES and OVERSAMPLE_FACTOR > 1.0:
    train_cmd += ["--oversample_classes", *OVERSAMPLE_CLASSES]
    train_cmd += ["--oversample_factor", str(OVERSAMPLE_FACTOR)]
if MAX_TRAIN_BATCHES > 0:
    train_cmd += ["--max_train_batches", str(MAX_TRAIN_BATCHES)]
if MAX_VAL_BATCHES > 0:
    train_cmd += ["--max_val_batches", str(MAX_VAL_BATCHES)]

print("Train command:")
print(" ".join(train_cmd))
subprocess.run(train_cmd, check=True)

history_files = sorted((PROJECT_DIR / "models").glob("train_history_*.csv"), key=lambda item: item.stat().st_mtime)
if history_files:
    HISTORY_CSV = history_files[-1]
    history_df = pd.read_csv(HISTORY_CSV)
    print("History CSV:", HISTORY_CSV)
    display_columns = [
        "epoch", "lr", "train_loss", "val_loss", "val_map50", "best_map50",
        "train_box_loss", "train_objectness_loss", "train_class_loss",
        "val_box_loss", "val_objectness_loss", "val_class_loss",
    ]
    display(history_df[[col for col in display_columns if col in history_df.columns]].tail(10))
else:
    HISTORY_CSV = None
    print("No train history CSV found.")


Train command:
/usr/bin/python3 train.py --train_data /kaggle/input/datasets/duong1589/object-detection/public/annotations/train.json --val_data /kaggle/input/datasets/duong1589/object-detection/public/annotations/val.json --image_dir /kaggle/input/datasets/duong1589/object-detection/public/train/images --val_image_dir /kaggle/input/datasets/duong1589/object-detection/public/val/images --checkpoint_dir /kaggle/working/object-detection/models --image_size 640 --epochs 100 --batch_size 8 --num_workers 2 --lr 0.0001 --weight_decay 0.0001 --conf_threshold 0.25 --nms_threshold 0.35 --amp --pretrained_backbone
device=cuda amp=True
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 155MB/s]


epoch=1/100 lr=6.66667e-05 train_loss=3.1677 val_loss=2.6194 val_map50=0.2738 best_map50=0.2738


epoch=2/100 lr=0.0001 train_loss=2.5917 val_loss=2.5024 val_map50=0.3187 best_map50=0.3187


epoch=3/100 lr=0.0001 train_loss=2.4079 val_loss=2.3149 val_map50=0.3617 best_map50=0.3617


...

epoch=97/100 lr=2.35829e-07 train_loss=0.9318 val_loss=1.9172 val_map50=0.6118 best_map50=0.6224


epoch=98/100 lr=1.04859e-07 train_loss=0.9261 val_loss=1.9211 val_map50=0.6086 best_map50=0.6224


epoch=99/100 lr=2.62215e-08 train_loss=0.9269 val_loss=1.9194 val_map50=0.6072 best_map50=0.6224


epoch=100/100 lr=0 train_loss=0.9272 val_loss=1.9146 val_map50=0.6111 best_map50=0.6224
best_checkpoint=/kaggle/working/object-detection/models/best.pth
history=/kaggle/working/object-detection/models/train_history_20260527_060713.csv
History CSV: /kaggle/working/object-detection/models/train_history_20260527_060713.csv


,epoch,lr,train_loss,val_loss,val_map50,best_map50,train_box_loss,train_objectness_loss,train_class_loss,val_box_loss,val_objectness_loss,val_class_loss
90,91,2.109134e-06,0.932093,1.911353,0.610142,0.622363,0.178833,0.033717,0.008426,0.288289,0.334056,0.271702
91,92,1.668957e-06,0.931259,1.915306,0.608753,0.622363,0.178499,0.033651,0.010230,0.288813,0.333267,0.275951
92,93,1.279474e-06,0.921804,1.924061,0.608018,0.622363,0.177570,0.030218,0.007474,0.288972,0.340516,0.277366
93,94,9.410912e-07,0.931132,1.912841,0.609193,0.622363,0.179066,0.032763,0.006075,0.288446,0.333297,0.274630
94,95,6.541646e-07,0.930809,1.920356,0.608375,0.622363,0.178725,0.033646,0.007080,0.289164,0.337944,0.273188
95,96,4.189949e-07,0.923441,1.916512,0.611716,0.622363,0.177594,0.032686,0.005567,0.288656,0.333912,0.278640
96,97,2.358289e-07,0.931804,1.917247,0.611826,0.622363,0.179197,0.032630,0.006381,0.288834,0.333126,0.279903
97,98,1.048587e-07,0.926066,1.921119,0.608578,0.622363,0.177806,0.033379,0.007316,0.288860,0.338171,0.277294
98,99,2.622155e-08,0.926885,1.919418,0.607201,0.622363,0.178483,0.031547,0.005847,0.288822,0.336822,0.276968
99,100,0.000000e+00,0.927176,1.914559,0.611063,0.622363,0.177952,0.034000,0.006832,0.288541,0.331542,0.280625


## 6. Predict On Validation And Evaluate

In [6]:
VAL_PREDICTIONS = PROJECT_DIR / "val_predictions.json"
predict_val_cmd = [
    sys.executable, "predict.py",
    "--image_dir", str(VAL_IMAGES),
    "--output", str(VAL_PREDICTIONS),
    "--checkpoint", str(PROJECT_DIR / "models" / "best.pth"),
    "--batch_size", str(BATCH_SIZE),
    "--conf_threshold", str(CONF_THRESHOLD),
    "--nms_threshold", str(NMS_THRESHOLD),
]
print("Predict validation command:")
print(" ".join(predict_val_cmd))
subprocess.run(predict_val_cmd, check=True)

val_predictions_data = json.loads(VAL_PREDICTIONS.read_text(encoding="utf-8"))
num_pred_boxes = sum(len(item["boxes"]) for item in val_predictions_data)
print(f"Validation predictions: images={len(val_predictions_data)} boxes={num_pred_boxes}")
print("Prediction preview:")
print(json.dumps(val_predictions_data[:3], ensure_ascii=False, indent=2))

if EVALUATOR.exists():
    VAL_SCORE = PROJECT_DIR / "val_score.json"
    subprocess.run([
        sys.executable, str(EVALUATOR),
        "--ground_truth", str(VAL_JSON),
        "--predictions", str(VAL_PREDICTIONS),
        "--output", str(VAL_SCORE),
    ], check=True)
    score = json.loads(VAL_SCORE.read_text(encoding="utf-8"))
    print("Validation score:")
    print(json.dumps(score, ensure_ascii=False, indent=2))
    if "per_class" in score:
        per_class_df = pd.DataFrame.from_dict(score["per_class"], orient="index")
        display(per_class_df)
else:
    VAL_SCORE = None
    print("Evaluator not found, skipped local validation scoring.")


Predict validation command:
/usr/bin/python3 predict.py --image_dir /kaggle/input/datasets/duong1589/object-detection/public/val/images --output /kaggle/working/object-detection/val_predictions.json --checkpoint /kaggle/working/object-detection/models/best.pth --batch_size 8 --conf_threshold 0.25 --nms_threshold 0.35
wrote 1500 predictions to /kaggle/working/object-detection/val_predictions.json
Validation predictions: images=1500 boxes=1955
Prediction preview:
[
  {
    "image_id": "img_00090df9158f.jpg",
    "boxes": [
      {
        "class": "dog",
        "confidence": 0.535855,
        "bbox": [
          136.13,
          11.42,
          500.0,
          368.65
        ]
      }
    ]
  },
  {
    "image_id": "img_003c3ddc6a82.jpg",
    "boxes": [
      {
        "class": "dog",
        "confidence": 0.435268,
        "bbox": [
          4.61,
          65.9,
          496.64,
          359.09
        ]
      }
    ]
  },
  {
    "image_id": "img_01073356e41c.jpg",
    "boxes":

,ap,num_ground_truth,num_predictions,true_positives,false_positives,recall,precision
person,0.742156,1074,1076,838,238,0.780261,0.778810
car,0.615791,283,271,186,85,0.657244,0.686347
dog,0.608077,206,191,139,52,0.674757,0.727749
cat,0.733786,176,178,136,42,0.772727,0.764045
chair,0.419132,282,239,140,99,0.496454,0.585774


## 7. Predict On Hidden/Test Images If Available

In [7]:
if TEST_IMAGE_DIR is None:
    print("TEST_IMAGE_DIR is None. Skipped final prediction.")
else:
    test_dir = Path(TEST_IMAGE_DIR)
    final_cmd = [
        sys.executable, "predict.py",
        "--image_dir", str(test_dir),
        "--output", str(FINAL_PREDICTIONS),
        "--checkpoint", str(PROJECT_DIR / "models" / "best.pth"),
        "--batch_size", str(BATCH_SIZE),
        "--conf_threshold", str(CONF_THRESHOLD),
        "--nms_threshold", str(NMS_THRESHOLD),
    ]
    print("Final predict command:")
    print(" ".join(final_cmd))
    subprocess.run(final_cmd, check=True)
    final_data = json.loads(FINAL_PREDICTIONS.read_text(encoding="utf-8"))
    print(f"Final predictions: images={len(final_data)} boxes={sum(len(item['boxes']) for item in final_data)}")
    print("Final prediction preview:")
    print(json.dumps(final_data[:3], ensure_ascii=False, indent=2))
    print("Final predictions path:", FINAL_PREDICTIONS)


TEST_IMAGE_DIR is None. Skipped final prediction.


## 8. Collect Artifacts

In [8]:
import shutil

ARTIFACT_DIR = Path("/kaggle/working/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    PROJECT_DIR / "models" / "best.pth",
    PROJECT_DIR / "models" / "last.pth",
    VAL_PREDICTIONS,
    PROJECT_DIR / "val_score.json",
]
files_to_copy += sorted((PROJECT_DIR / "models").glob("train_history_*.csv"))
if FINAL_PREDICTIONS.exists():
    files_to_copy.append(FINAL_PREDICTIONS)

copied = []
for src in files_to_copy:
    if src.exists():
        dst = ARTIFACT_DIR / src.name
        shutil.copy2(src, dst)
        copied.append(dst)

zip_path = shutil.make_archive("/kaggle/working/object_detection_artifacts", "zip", ARTIFACT_DIR)
print("Artifacts directory:", ARTIFACT_DIR)
print("Artifacts zip:", zip_path)
print("Files:")
artifact_rows = []
for path in sorted(ARTIFACT_DIR.iterdir()):
    artifact_rows.append({"file": path.name, "size_mb": round(path.stat().st_size / (1024 * 1024), 3)})
display(pd.DataFrame(artifact_rows))


Artifacts directory: /kaggle/working/artifacts
Artifacts zip: /kaggle/working/object_detection_artifacts.zip
Files:


,file,size_mb
0,best.pth,191.605
1,last.pth,191.605
2,train_history_20260523_023548.csv,0.001
3,train_history_20260527_060713.csv,0.061
4,val_predictions.json,0.417
5,val_score.json,0.001


## 9. Summary Shown In Notebook

In [9]:
summary = {
    "best_checkpoint": str(PROJECT_DIR / "models" / "best.pth"),
    "history_csv": str(HISTORY_CSV) if HISTORY_CSV is not None else None,
    "val_predictions": str(VAL_PREDICTIONS),
    "val_score": str(PROJECT_DIR / "val_score.json"),
    "artifacts_zip": "/kaggle/working/object_detection_artifacts.zip",
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

if HISTORY_CSV is not None:
    history_df = pd.read_csv(HISTORY_CSV)
    print("Final history row:")
    display(history_df.tail(1))

score_path = PROJECT_DIR / "val_score.json"
if score_path.exists():
    score = json.loads(score_path.read_text(encoding="utf-8"))
    print(f"mAP@0.5={score.get('mAP@0.5')} performance_points={score.get('performance_points')}")


{
  "best_checkpoint": "/kaggle/working/object-detection/models/best.pth",
  "history_csv": "/kaggle/working/object-detection/models/train_history_20260527_060713.csv",
  "val_predictions": "/kaggle/working/object-detection/val_predictions.json",
  "val_score": "/kaggle/working/object-detection/val_score.json",
  "artifacts_zip": "/kaggle/working/object_detection_artifacts.zip"
}
Final history row:


,run_started_at,epoch,total_epochs,train_data,val_data,image_dir,val_image_dir,image_size,batch_size,num_workers,...,train_objectness_loss,train_class_loss,train_num_positive,val_loss,val_box_loss,val_objectness_loss,val_class_loss,val_num_positive,val_map50,best_map50
99,2026-05-27 06:07:13,100,100,/kaggle/input/datasets/duong1589/object-detect...,/kaggle/input/datasets/duong1589/object-detect...,/kaggle/input/datasets/duong1589/object-detect...,/kaggle/input/datasets/duong1589/object-detect...,640,8,2,...,0.034,0.006832,92.948827,1.914559,0.288541,0.331542,0.280625,90.425532,0.611063,0.622363


mAP@0.5=0.623789 performance_points=20
